# 13: OWL Profile Classification & Ontological Similarity

Inspects the output of the OWL 2 RL classification pipeline (`scripts/run_owl_classification.py`).

**Covers:**
- Classification distribution across the 117 defined profile classes (only purpose, navigation and engagement classes are assigned in the evaluated KGs; see Known issues in the README)
- IC weighting and class informativeness
- Event arc patterns
- Ontological similarity: find similar trajectories, pairwise matrix
- POI recommendations from archetype similarity

**Prerequisites:** Run `python scripts/run_owl_classification.py` before this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helpers.ontological_recommender import OntologicalRecommender, LOW_IC_THRESHOLD
from helpers.fuseki_config import query_endpoint

rec = OntologicalRecommender()
rec.load_classifications()
print(f"Trajectories: {rec.total_trajectories}")
print(f"Classes: {len(rec.class_counts)}")

## 1. Classification Distribution

In [ ]:
# Class distribution with IC weights
dist = rec.class_distribution()
df_dist = pd.DataFrame(dist, columns=["class", "count", "IC"])
df_dist["pct"] = (df_dist["count"] / rec.total_trajectories * 100).round(1)
df_dist["informative"] = df_dist["IC"] >= LOW_IC_THRESHOLD

# Assign class type based on naming
def class_type(name):
    if "Focused" in name: return "Focused"
    if "Frequent" in name: return "User Frequency"
    if "Arc" in name: return "Event Arc"
    if name.startswith("Has"): return "Interaction/Event"
    if "Immersive" in name and "Visitor" in name: return "Immersive Visitor"
    if "Engagement" in name: return "Engagement"
    if "Visitor" in name: return "Visitor"
    if any(x in name for x in ["Targeted", "Casual", "Explorer"]): return "Purpose-Nav"
    return "Other"

df_dist["type"] = df_dist["class"].apply(class_type)
print(f"Total type assertions: {df_dist['count'].sum()}")
print(f"Informative classes (IC >= {LOW_IC_THRESHOLD}): {df_dist['informative'].sum()}/{len(df_dist)}")
print()
df_dist

In [ ]:
# Horizontal bar chart — informative classes only
df_info = df_dist[df_dist["informative"]].sort_values("count")

fig, ax = plt.subplots(figsize=(10, max(6, len(df_info) * 0.28)))
colors = {
    "Visitor": "#4C72B0", "Purpose-Nav": "#DD8452", "Immersive Visitor": "#55A868",
    "Engagement": "#C44E52", "Focused": "#8172B3", "Event Arc": "#937860",
    "Interaction/Event": "#DA8BC3", "Other": "#8C8C8C", "User Frequency": "#CCB974",
}
bar_colors = [colors.get(t, "#8C8C8C") for t in df_info["type"]]
ax.barh(df_info["class"], df_info["count"], color=bar_colors)
ax.set_xlabel("Trajectories")
ax.set_title(f"Profile Class Distribution (informative only, IC >= {LOW_IC_THRESHOLD})")

# Legend
from matplotlib.patches import Patch
used_types = df_info["type"].unique()
legend_elements = [Patch(facecolor=colors.get(t, "#8C8C8C"), label=t) for t in sorted(used_types)]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 2. Classifications Per Trajectory

In [ ]:
# How many classes per trajectory?
class_counts_per_traj = [len(c) for c in rec.classifications.values()]
informative_per_traj = [
    sum(1 for cls in c if rec.ic_weights.get(cls, 0) >= LOW_IC_THRESHOLD)
    for c in rec.classifications.values()
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(class_counts_per_traj, bins=range(0, max(class_counts_per_traj) + 2), edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Classes per trajectory")
axes[0].set_ylabel("Count")
axes[0].set_title(f"All classes (mean={np.mean(class_counts_per_traj):.1f})")

axes[1].hist(informative_per_traj, bins=range(0, max(informative_per_traj) + 2), edgecolor="black", alpha=0.7, color="#DD8452")
axes[1].set_xlabel("Informative classes per trajectory")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Informative only (mean={np.mean(informative_per_traj):.1f})")

plt.tight_layout()
plt.show()

# Trajectories with 0 informative classes
zero_info = sum(1 for x in informative_per_traj if x == 0)
print(f"Trajectories with 0 informative classes: {zero_info}/{len(informative_per_traj)}")
print(f"Unclassified entirely: {543 - rec.total_trajectories}")

## 3. Trajectory Detail — Richest and Sparsest Profiles

In [ ]:
# Richest trajectories
by_count = sorted(rec.classifications.items(), key=lambda x: len(x[1]), reverse=True)

print("=== Top 5 richest trajectories ===")
for uri, classes in by_count[:5]:
    name = uri.split("#")[-1]
    informative = sorted(c for c in classes if rec.ic_weights.get(c, 0) >= LOW_IC_THRESHOLD)
    print(f"\n{name} ({len(classes)} total, {len(informative)} informative):")
    for cls in informative:
        print(f"  {cls} (IC={rec.ic_weights[cls]:.2f})")

print("\n\n=== Sparsest classified trajectories (1-3 classes) ===")
for uri, classes in sorted(rec.classifications.items(), key=lambda x: len(x[1]))[:5]:
    name = uri.split("#")[-1]
    print(f"{name}: {sorted(classes)}")

## 4. Ontological Similarity

In [ ]:
# Pick the richest trajectory and find similar ones
target_uri = by_count[0][0]
target_name = target_uri.split("#")[-1]
print(f"Target: {target_name}")
print(f"Classes: {sorted(rec.classifications[target_uri])}\n")

similar = rec.find_similar(target_uri, top_k=10, vector_rerank=False)
print("Top 10 similar trajectories:")
for s in similar:
    name = s["uri"].split("#")[-1]
    top3 = ", ".join(s["shared_classes"][:3])
    print(f"  {name}: score={s['score']:.3f}, {s['n_shared']} shared [{top3}...]")

## 5. Pairwise Similarity Matrix

In [ ]:
result = rec.pairwise_similarity_matrix()
matrix = result["matrix"]
uris = result["uris"]

# Sort trajectories by dominant class for visual clustering
# Use the most specific (highest IC) class per trajectory as sort key
def sort_key(uri):
    classes = rec.classifications.get(uri, set())
    informative = [(c, rec.ic_weights.get(c, 0)) for c in classes if rec.ic_weights.get(c, 0) >= LOW_IC_THRESHOLD]
    if not informative:
        return ("zzz", 0)
    best = max(informative, key=lambda x: x[1])
    return best

sort_order = sorted(range(len(uris)), key=lambda i: sort_key(uris[i]))
sorted_matrix = matrix[np.ix_(sort_order, sort_order)]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sorted_matrix, cmap="YlOrRd", vmin=0, vmax=0.8, aspect="auto")
ax.set_title(f"Pairwise Ontological Similarity ({len(uris)} trajectories)")
ax.set_xlabel("Trajectory (sorted by most specific class)")
ax.set_ylabel("Trajectory")
plt.colorbar(im, ax=ax, label="IC-weighted Jaccard")
plt.tight_layout()
plt.show()

# Similarity distribution
upper = matrix[np.triu_indices_from(matrix, k=1)]
nonzero = upper[upper > 0]
print(f"Pairs: {len(upper)}, non-zero: {len(nonzero)} ({len(nonzero)/len(upper)*100:.1f}%)")
print(f"Mean (non-zero): {nonzero.mean():.3f}, median: {np.median(nonzero):.3f}, max: {nonzero.max():.3f}")

## 6. POI Recommendations

In [ ]:
# Recommend POIs for the richest trajectory
recs = rec.recommend_pois(target_uri, top_k=10, sim_pool=20)

print(f"POI Recommendations for {target_name}:")
print(f"Target classes: {sorted(c for c in rec.classifications[target_uri] if rec.ic_weights.get(c,0) >= LOW_IC_THRESHOLD)}\n")

if recs:
    df_recs = pd.DataFrame([{
        "POI": r["poi_uri"].split("#")[-1],
        "Type": r["poi_type"],
        "Score": r["score"],
        "Frequency": f"{r['frequency']}/20",
        "Best Interaction": r["best_interaction"],
    } for r in recs])
    display(df_recs)
else:
    print("No recommendations (insufficient similar trajectories with unvisited POIs)")

## 7. Event Distribution

In [ ]:
from helpers.fuseki_queries import execute_sparql_query

# Event counts from KG
result = execute_sparql_query(query_endpoint, """
PREFIX events: <http://w3id.org/trajecto/events#>
SELECT ?type (COUNT(*) AS ?count) WHERE {
  ?evt a ?type .
  FILTER(STRSTARTS(STR(?type), "http://w3id.org/trajecto/events#"))
}
GROUP BY ?type
ORDER BY DESC(?count)
""")

print("Materialized events (SR-50..54):")
for b in result["results"]["bindings"]:
    name = b["type"]["value"].split("#")[-1]
    count = b["count"]["value"]
    print(f"  {name}: {count}")

# Event arc classes from inferred graph
print("\nEvent arc profile classes:")
event_classes = [c for c in rec.class_counts if any(x in c for x in ["Arc", "Escalation", "Drop", "Discovery", "Transition", "Encounter"])]
for cls in sorted(event_classes, key=lambda c: -rec.class_counts[c]):
    print(f"  {cls}: {rec.class_counts[cls]} trajectories (IC={rec.ic_weights[cls]:.2f})")